In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from google.colab import drive
drive.mount('/content/drive')
url26 = "https://raw.githubusercontent.com/transientlunatic/Rugby-Data/refs/heads/master/json/pro-d2-2025-2026.json"
df26 = pd.read_json(url26)


Mounted at /content/drive


In [ ]:
# 1. On part de ton df26 brut
df = df26.copy()

# --- A. EXTRACTION DES DONNÉES ---
def extraire_infos(cellule):
    if not isinstance(cellule, dict):
        return pd.Series([None, 0, 0])
    equipe = cellule.get('team')
    score = cellule.get('score', 0)
    essais = 0
    if 'scores' in cellule and isinstance(cellule['scores'], list):
        for action in cellule['scores']:
            if action.get('type') in ['Try', 'Penalty Try']:
                essais += 1
    return pd.Series([equipe, score, essais])

# Extraction
df[['equipe_domicile', 'home_score', 'home_tries']] = df['home'].apply(extraire_infos)
df[['equipe_exterieur', 'away_score', 'away_tries']] = df['away'].apply(extraire_infos)

# --- B. CALCUL DES POINTS (PRO D2) ---
def calculer_points_pro_d2(row):
    if pd.isna(row['home_score']) or pd.isna(row['away_score']):
        return pd.Series([0, 0, 0, 0, 0, 0])

    score_dom = int(row['home_score'])
    score_ext = int(row['away_score'])
    essais_dom = int(row['home_tries'])
    essais_ext = int(row['away_tries'])

    pts_dom, pts_ext = 0, 0
    bonus_off_dom, bonus_off_ext = 0, 0
    bonus_def_dom, bonus_def_ext = 0, 0

    if score_dom > score_ext: pts_dom = 4
    elif score_dom < score_ext: pts_ext = 4
    else: pts_dom = 2; pts_ext = 2

    if essais_dom - essais_ext >= 3: bonus_off_dom = 1
    if essais_ext - essais_dom >= 3: bonus_off_ext = 1

    ecart = abs(score_dom - score_ext)
    if ecart <= 5:
        if score_dom < score_ext: bonus_def_dom = 1
        elif score_ext < score_dom: bonus_def_ext = 1

    total_dom = pts_dom + bonus_off_dom + bonus_def_dom
    total_ext = pts_ext + bonus_off_ext + bonus_def_ext

    return pd.Series([total_dom, total_ext, bonus_off_dom, bonus_off_ext, bonus_def_dom, bonus_def_ext])

# Application
colonnes_calculs = ['pts_domicile', 'pts_exterieur', 'bonus_off_dom', 'bonus_off_ext', 'bonus_def_dom', 'bonus_def_ext']
df[colonnes_calculs] = df.apply(calculer_points_pro_d2, axis=1)

# --- C. CLASSEMENT RÉEL ---
dom = df[['equipe_domicile', 'pts_domicile']].copy()
dom.columns = ['Equipe', 'Points']
ext = df[['equipe_exterieur', 'pts_exterieur']].copy()
ext.columns = ['Equipe', 'Points']

classement_reel = pd.concat([dom, ext]).groupby('Equipe')['Points'].sum().reset_index()
classement_reel = classement_reel.sort_values(by='Points', ascending=False).reset_index(drop=True)
classement_reel['Rang_Reel'] = classement_reel.index + 1

# --- D. CLASSEMENT ATTENDU (Modèle simplifié) ---
moyenne_points_generale = df[['home_score', 'away_score']].mean().mean()

stats_dom = df.groupby('equipe_domicile').agg(points_marques_dom=('home_score', 'mean'), points_encaisses_dom=('away_score', 'mean')).reset_index()
stats_ext = df.groupby('equipe_exterieur').agg(points_marques_ext=('away_score', 'mean'), points_encaisses_ext=('home_score', 'mean')).reset_index()

stats = pd.merge(stats_dom, stats_ext, left_on='equipe_domicile', right_on='equipe_exterieur', how='outer')
stats['Equipe'] = stats['equipe_domicile'].fillna(stats['equipe_exterieur'])
stats = stats.drop(columns=['equipe_domicile', 'equipe_exterieur'])

stats['force_attaque'] = (stats['points_marques_dom'] + stats['points_marques_ext']) / 2
stats['force_defense'] = (stats['points_encaisses_dom'] + stats['points_encaisses_ext']) / 2

points_attendus = {equipe: 0 for equipe in stats['Equipe']}

for index, row in df.iterrows():
    dom_equipe = row['equipe_domicile']
    ext_equipe = row['equipe_exterieur']

    # Approximation des points attendus basés sur la force (plus l'équipe est forte, plus elle gagne de points attendus)
    points_attendus[dom_equipe] += (row['pts_domicile'] * 0.7) + (row['pts_exterieur'] * 0.3)
    points_attendus[ext_equipe] += (row['pts_exterieur'] * 0.7) + (row['pts_domicile'] * 0.3)

df_attendu = pd.DataFrame(list(points_attendus.items()), columns=['Equipe', 'Points_Attendus'])
df_attendu = df_attendu.sort_values(by='Points_Attendus', ascending=False).reset_index(drop=True)
df_attendu['Rang_Attendu'] = df_attendu.index + 1

# --- E. FUSION FINALE ---
resultat_final = pd.merge(classement_reel, df_attendu, on='Equipe')

print("Classement Réel vs Classement Attendu (Saison 2025/2026) :")
display(resultat_final[['Rang_Reel', 'Equipe', 'Points', 'Rang_Attendu', 'Points_Attendus']].head(16))

Classement Réel vs Classement Attendu (Saison 2025/2026) :


,Rang_Reel,Equipe,Points,Rang_Attendu,Points_Attendus
0,1,Vannes,99,1,75.0
1,2,Colomiers,81,2,66.6
2,3,Provence,71,3,65.0
3,4,Valence-Romans,71,6,62.0
4,5,Brive,69,5,62.7
5,6,Oyonnax,67,4,63.1
6,7,Agen,57,7,57.9
7,8,Dax,54,8,57.0
8,9,Aurillac,52,9,55.9
9,10,Soyaux Angouleme,50,10,55.4


In [ ]:
classement_reel.to_csv('/content/drive/MyDrive/Rugby_ProD2_Forecast/classement_reel_N-1.csv', index=False)
df.to_csv('/content/drive/MyDrive/Rugby_ProD2_Forecast/matchs_N-1.csv', index=False)